# Fase 4 y 5 — Visualización

**Fase 4:** Boxplots de HV, GD e IGD (original vs modificado) por N, y frentes de Pareto superpuestos.

**Fase 5:** Figura del frente de Pareto para el paper — corrida mediana (N=100, same_seed), exportada en PDF vectorial.

In [ ]:
import os
import glob
import pickle
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype']  = 42

RESULTS_BASE = './results'
FIGURES_DIR  = './results/figures'
N_VALUES     = [50, 100, 150, 200]
METRICS      = ['HV', 'GD', 'IGD']
VARIANTE_LABELS = {'original': 'HP-MOEA', 'modified': 'HP-MOEA Mod.'}
COLORS = {'original': '#2196F3', 'modified': '#FF5722'}

os.makedirs(FIGURES_DIR, exist_ok=True)

In [ ]:
metrics_path = f'{RESULTS_BASE}/metrics.csv'
if not os.path.exists(metrics_path):
    raise FileNotFoundError('Ejecutar metrics.ipynb primero para generar metrics.csv')

df = pd.read_csv(metrics_path)
print(f'Corridas cargadas: {len(df)}')
df.groupby(['variante', 'seed_mode', 'N']).size().rename('n').to_frame()

## Fase 4a — Boxplots de métricas por N y modo de semilla

In [ ]:
def plot_metric_boxplots(df, seed_mode, metric, ax):
    sub = df[df.seed_mode == seed_mode].copy()
    sub['N_str'] = sub['N'].astype(str)
    sub['label'] = sub['variante'].map(VARIANTE_LABELS)

    if sub.empty:
        ax.text(0.5, 0.5, f'Sin datos\n({seed_mode})', ha='center', va='center',
                transform=ax.transAxes, fontsize=10, color='gray')
        label = 'mismas semillas' if seed_mode == 'same_seed' else 'semillas distintas'
        ax.set_title(f'{metric} — {label}')
        return

    palette = {VARIANTE_LABELS[v]: COLORS[v] for v in VARIANTE_LABELS if v in sub.variante.values}
    sns.boxplot(
        data=sub, x='N_str', y=metric, hue='label',
        order=[str(n) for n in N_VALUES if str(n) in sub.N_str.values],
        palette=palette, ax=ax,
        linewidth=0.8, flierprops=dict(marker='o', markersize=3)
    )
    ax.set_xlabel('N (tamaño de población)')
    ax.set_ylabel(metric)
    label = 'mismas semillas' if seed_mode == 'same_seed' else 'semillas distintas'
    ax.set_title(f'{metric} — {label}')
    ax.legend(title=None, fontsize=8)

for metric in METRICS:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=False)
    plot_metric_boxplots(df, 'same_seed', metric, axes[0])
    plot_metric_boxplots(df, 'diff_seed', metric, axes[1])
    fig.suptitle(f'Comparacion {metric}: HP-MOEA original vs modificado', fontsize=12, fontweight='bold')
    plt.tight_layout()
    out = os.path.join(FIGURES_DIR, f'boxplot_{metric.lower()}.pdf')
    plt.savefig(out, bbox_inches='tight')
    print(f'Guardado: {out}')
    plt.show()

## Fase 4b — Frentes de Pareto superpuestos por N

In [ ]:
def load_fronts(variante, seed_mode, n):
    folder = os.path.join(RESULTS_BASE, variante, seed_mode, f'N{n}')
    if not os.path.exists(folder):
        return []
    fronts = []
    for pkl_path in sorted(glob.glob(os.path.join(folder, '*.pkl'))):
        with open(pkl_path, 'rb') as f:
            data = pickle.load(f)
        fronts.append(np.array(data['fitnessF']))
    return fronts

In [ ]:
for seed_mode in ['same_seed', 'diff_seed']:
    available_N = [n for n in N_VALUES
                   if load_fronts('original', seed_mode, n) or load_fronts('modified', seed_mode, n)]
    if not available_N:
        print(f'Sin datos para {seed_mode}')
        continue

    fig, axes = plt.subplots(1, len(available_N), figsize=(5 * len(available_N), 4))
    if len(available_N) == 1:
        axes = [axes]

    for ax, n in zip(axes, available_N):
        for variante in ['original', 'modified']:
            fronts = load_fronts(variante, seed_mode, n)
            if not fronts:
                continue
            all_pts = np.vstack(fronts)
            ax.scatter(-all_pts[:, 0], -all_pts[:, 1],
                       s=4, alpha=0.25, color=COLORS[variante],
                       label=VARIANTE_LABELS[variante])
        ax.set_title(f'N = {n}')
        ax.set_xlabel('Ganancia (profit)')
        ax.set_ylabel('Novedad (novelty)')
        ax.legend(fontsize=7, markerscale=3)

    label_sm = 'mismas semillas' if seed_mode == 'same_seed' else 'semillas distintas'
    fig.suptitle(f'Frentes de Pareto — {label_sm}', fontsize=12, fontweight='bold')
    plt.tight_layout()
    out = os.path.join(FIGURES_DIR, f'pareto_fronts_{seed_mode}.pdf')
    plt.savefig(out, bbox_inches='tight')
    print(f'Guardado: {out}')
    plt.show()

## Fase 5 — Frente de Pareto para el paper (N=100, same_seed, corrida mediana)

Se selecciona la corrida cuyo HV es la **mediana** de las 30 corridas N=100 misma semilla,
de manera independiente para original y modificado.
Scatter superpuesto exportado en PDF vectorial.

In [ ]:
def find_median_run(df, variante, seed_mode='same_seed', n=100, metric='HV'):
    sub = df[(df.variante == variante) & (df.seed_mode == seed_mode) & (df.N == n)].copy()
    if sub.empty:
        return None
    median_val = sub[metric].median()
    idx = (sub[metric] - median_val).abs().idxmin()
    return sub.loc[idx]

def load_front_by_corrida(row):
    folder = os.path.join(RESULTS_BASE, row['variante'], row['seed_mode'], f'N{int(row["N"])}')
    pkls   = sorted(glob.glob(os.path.join(folder, '*.pkl')))
    corrida_idx = int(row['corrida']) - 1
    if corrida_idx >= len(pkls):
        return None
    with open(pkls[corrida_idx], 'rb') as f:
        data = pickle.load(f)
    return np.array(data['fitnessF'])

row_orig = find_median_run(df, 'original')
row_mod  = find_median_run(df, 'modified')

if row_orig is not None:
    print(f"Original   — corrida {int(row_orig['corrida'])}, HV = {row_orig['HV']:.6f}")
else:
    print('Original   — sin datos para N=100 same_seed')

if row_mod is not None:
    print(f"Modificado — corrida {int(row_mod['corrida'])}, HV = {row_mod['HV']:.6f}")
else:
    print('Modificado — sin datos para N=100 same_seed')

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
plotted = False

for row, variante in [(row_orig, 'original'), (row_mod, 'modified')]:
    if row is None:
        continue
    F = load_front_by_corrida(row)
    if F is None:
        continue
    pts = np.column_stack((-F[:, 0], -F[:, 1]))
    pts = pts[pts[:, 0].argsort()]
    ax.scatter(pts[:, 0], pts[:, 1],
               s=18, color=COLORS[variante], alpha=0.85,
               label=VARIANTE_LABELS[variante], edgecolors='none')
    plotted = True

if plotted:
    ax.set_xlabel('Ganancia (profit)', fontsize=11)
    ax.set_ylabel('Novedad (novelty)', fontsize=11)
    ax.set_title('Frente de Pareto — N=100, corrida mediana', fontsize=11)
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    out = os.path.join(FIGURES_DIR, 'pareto_paper_N100_mediana.pdf')
    plt.savefig(out, bbox_inches='tight', format='pdf')
    print(f'Guardado: {out}')
    plt.show()
else:
    ax.text(0.5, 0.5, 'Sin datos para N=100 same_seed',
            ha='center', va='center', transform=ax.transAxes, color='gray')
    plt.show()
    print('No hay datos. Completar corridas N=100 same_seed para original y modificado.')